# EconEnv on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/merwanroudane/econenv/blob/main/examples/11_colab_quickstart.ipynb)

**Python and R in one Colab notebook**, sharing a dataset with no CSV in
between.

---

**Developed by Dr Merwan Roudane**  
GitHub: <https://github.com/merwanroudane>  
Package: <https://pypi.org/project/econenv/>  
Repository: <https://github.com/merwanroudane/econenv>

---

## What works on Colab, and what cannot

Colab runs on Linux, and that decides this before anything is installed:

| Engine | On Colab | Why |
|---|---|---|
| **Python** | works | it is the kernel |
| **R** | works | R is already on the Colab image |
| **Stata** | no | commercial, not installed, and the runtime is destroyed at the end of the session |
| **EViews** | no | its automation interface is Windows COM; there is no Linux version of it |

This is a limitation of the programs, not of EconEnv, and no amount of
configuration changes it. For Stata and EViews, run
[the full four-engine notebook](https://github.com/merwanroudane/econenv/blob/main/examples/10_real_data_four_engines.ipynb)
on a local Windows machine.

`%econ doctor` says all of this for you, on the machine you are actually on.

## 1. Install

One line. `%pip` installs into the kernel that is running, which is what
you want inside a notebook.

In [ ]:
%pip install -q econenv

## 2. Load and check

The status table shows what EconEnv found. On Colab you should see Python
and R configured, and EViews reported as unavailable — which is correct,
not a problem to solve.

In [ ]:
%load_ext econenv

In [ ]:
%econ status

`%econ doctor` explains anything that is missing, and on Colab it says
explicitly why Stata and EViews cannot be there.

In [ ]:
%econ doctor

## 3. Python — build a dataset

Real US quarterly macroeconomic data, 1959Q1–2009Q3, shipped with
statsmodels — so nothing is downloaded and nothing is private.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import econenv

raw = sm.datasets.macrodata.load_pandas().data
idx = pd.PeriodIndex(
    year=raw.year.astype(int), quarter=raw.quarter.astype(int), freq='Q'
).to_timestamp()

macro = pd.DataFrame(
    {
        'lrgdp': np.log(raw.realgdp.values),
        'lrcons': np.log(raw.realcons.values),
        'realint': raw.realint.values,
    },
    index=idx,
)
macro.index.name = 'date'
print(len(macro), 'quarters')
macro.head()

## 4. R — the same data, no file in between

`-i macro` sends the DataFrame into R. It arrives as a real `data.frame`;
the pandas index becomes a `date` column, because R has no separate notion
of an index.

In [ ]:
%%R -i macro
cat('R received', nrow(macro), 'rows\n')
str(macro)

In [ ]:
%%R
fit <- lm(lrcons ~ lrgdp + realint, data = macro)
summary(fit)

R's diagnostic plots render straight into the Colab output cell:

In [ ]:
%%R
par(mfrow = c(2, 2))
plot(fit)

### Bring results back to Python

`-o` returns an R object to the Python side.

In [ ]:
%%R -o coefs
coefs <- as.data.frame(summary(fit)$coefficients)

In [ ]:
coefs

## 5. Compare Python and R on the same model

`compare_ols` runs the specification in every engine available. On Colab
that is Python and R; on a Windows machine with all four installed, the
same line returns four columns.

In [ ]:
cmp = econenv.compare_ols(macro, 'lrcons ~ lrgdp + realint')
cmp

In [ ]:
cmp.coefficients()

The coefficients agree to machine precision. Any information criteria
that differ are reported with the reason — statsmodels uses $-2\ell + 2k$
while R counts $\sigma^2$ as a parameter — rather than being quietly
reconciled.

## 6. Record what produced this

Colab runtimes are disposable, which makes a snapshot more useful here
than anywhere else: it pins the versions that generated these numbers.

In [ ]:
econenv.snapshot()

---

## Next

- [The full four-engine notebook](https://github.com/merwanroudane/econenv/blob/main/examples/10_real_data_four_engines.ipynb) — Python, R, Stata **and** EViews, for a local Windows machine
- [Documentation site](https://merwanroudane.github.io/econenv/)
- [Installation & User Guide (PDF)](https://github.com/merwanroudane/econenv/blob/main/docs/guide/econenv-guide.pdf)
- [EViews commands for GUI users](https://github.com/merwanroudane/econenv/blob/main/docs/engines/eviews-commands.md)

*EconEnv — Dr Merwan Roudane — <https://github.com/merwanroudane/econenv>*